# 1 · The request → response exchange, and how a request exposes tools

Every agent loop starts with a plain HTTP exchange: send one JSON object, get
one JSON object back. Nothing magical happens here — no state, no memory. The
entire secret of an agent is **what goes into that request** and **what you do
with the reply**.

## The request

A Chat Completions request is roughly:

```json
{
  "model": "gpt-4o-mini",
  "messages": [ { "role": "system", "content": "..." },
                { "role": "user",   "content": "..." } ],
  "tools": [ { "type": "function", "function": { "name": "...", "description": "...", "parameters": {...} } } ],
  "tool_choice": "auto"
}
```

Two things matter for an *agent*:

- **`messages`** — the conversation, including a `system` message that sets the
  agent's behaviour. This is where the "system message format" lives.
- **`tools`** — the surface of what the model *may* call. **This is how a
  request exposes tools**: each tool is described as a JSON Schema the model
  can read, so the model knows it exists and how to fill in its arguments.

Below we build the exact request this runbook's agent sends, and print it as the
bytes that go over the wire. You will literally *see* the file, web and
calculator tools advertised to the model.


In [2]:
:dep agent_loop = { path = "/home/christian/Sandbox/agent-loop" }
:dep serde_json = "1"

// Load the crate pieces we need.
use agent_loop::chat::{ChatRequest, ChatMessage, ChatClient, default_model};
use agent_loop::tools::{ToolRegistry, file_tools, web_tools, calc_tools};

// Build the built-in tool surface explicitly so we can inspect it:
// three families — file ops, web, and a calculator.
let mut registry = ToolRegistry::new();
for t in [file_tools::read_tool(), file_tools::write_tool(), file_tools::list_tool(),
          web_tools::fetch_tool(), web_tools::search_tool(),
          calc_tools::calc_tool()] {
    registry.register(t);
}
println!("Registered {} tools.", registry.len());

// The messages: a system message (the agent behaviour) + the user prompt.
let messages = vec![
    ChatMessage::system("You are a coding agent. You can read and write files, search the web, and do arithmetic."),
    ChatMessage::user("Summarise the files in my project and tell me what hello.rs contains."),
];

// Assemble the request exactly as the loop does.
let request = ChatRequest {
    model: default_model(),
    messages,
    tools: Some(registry.as_chat_tools()),
    tool_choice: Some(serde_json::json!("auto")),
    temperature: Some(0.2),
};
request


Registered 6 tools.


ChatRequest { model: "deepseek-v4-flash-0731", messages: [ChatMessage { role: System, content: Some("You are a coding agent. You can read and write files, search the web, and do arithmetic."), tool_calls: None, tool_call_id: None }, ChatMessage { role: User, content: Some("Summarise the files in my project and tell me what hello.rs contains."), tool_calls: None, tool_call_id: None }], tools: Some([ChatTool { tool_type: "function", function: ToolFunction { name: "file_read", description: "Read the full contents of a text file at the given absolute or relative path. Returns file contents, or an error if the file does not exist or is not readable.", parameters: Object {"additionalProperties": Bool(false), "properties": Object {"path": Object {"description": String("Absolute or workspace-relative path of the file to read."), "type": String("string")}}, "required": Array [String("path")], "type": String("object")} } }, ChatTool { tool_type: "function", function: ToolFunction { name: "file_w

## The exact bytes the model receives

`to_json_pretty()` returns the precise JSON that gets POSTed to
`/chat/completions`. Look at the **`tools`** array: the model is handed the
`name`, a human-readable `description`, and a JSON `parameters` schema for each
of `file_read`, `file_write`, `file_list`, `web_fetch`, `web_search`, `calc`.
That schema is the *contract* the model uses to produce arguments.


In [3]:

// Dump the request body as it is serialized on the wire.
println!("{}", agent_loop::chat::pretty(&request.to_json_pretty()));


{


  "messages": [


    {


      "content": "You are a coding agent. You can read and write files, search the web, and do arithmetic.",


      "role": "system"


...

      },


      "type": "function"


    }


  ]


}


### System message format

The `system` message is special: it is the *standing instructions* that frame
every turn. The format is simply one message with `"role": "system"`. Tools are
not "the system" — they are a separate `tools` array. The system message tells
the model *how to behave*, while the `tools` array tells it *what it can touch*.

Here is the same request, but only its messages, so the shape of the `system`
message is crystal clear:


In [4]:

// Just the messages, to isolate the system-message format.
for m in &request.messages {
    println!("{:?}: {}", m.role, m.content.as_deref().unwrap_or("<tool_calls>"));
}


System: You are a coding agent. You can read and write files, search the web, and do arithmetic.


User: Summarise the files in my project and tell me what hello.rs contains.


()

### A sample system message

A system message is just an object with `"role": "system"` and a `content`
string. It carries the agent's standing behaviour. Here is one built the same
way the loop builds it, then printed as the JSON the endpoint actually receives:


In [5]:

// Build one system message by hand.
let system_message = agent_loop::chat::ChatMessage::system(
    "You are a coding agent. You can read and write files, search the web, and do arithmetic. Be concise."
);
println!("{}", agent_loop::chat::pretty(&serde_json::to_value(&system_message).unwrap()));


{


  "content": "You are a coding agent. You can read and write files, search the web, and do arithmetic. Be concise.",


  "role": "system"


}


## The response

Now send it live. Two things can happen:

- The endpoint answers with **text** (`finish_reason = "stop"`): the model is
  done.
- The endpoint answers with **`tool_calls`** (`finish_reason = "tool_calls"`):
  the model wants us to run tools. Note the model **never runs them itself** —
  it only *declares* the calls. Running them is the loop's job — demo 2 executes
  a single call, then parallel calls; demo 3 shows how a tool is added.

The cell below attempts the live exchange. If no endpoint is reachable it prints
a friendly note instead of failing the notebook.


In [3]:

// A tiny request with NO tools, so we see the plain text-exchange shape first.
let simple = agent_loop::chat::ChatClient::from_env();
match simple.complete_raw(&agent_loop::chat::simple_request(
        &agent_loop::chat::default_model(),
        "Answer in one short sentence.",
        "Say hello.")) {
    Ok(v) => println!("{}", agent_loop::chat::pretty(&v)),
    Err(e) => println!("[no reachable endpoint] live response skipped.
  {e}
  -> set OPENAI_BASE_URL and re-run this cell."),
}


{
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "Hello!",
        "role": "assistant"
      }
    }
  ],
  "created": 1789399829,
  "id": "chatcmpl-RSc8PXUMl9EgnpP4Se7c6Kxr",
  "model": "deepseek-v4-flash-0731",
  "object": "chat.completion",
  "usage": {
    "completion_tokens": 3,
    "prompt_tokens": 13,
    "total_tokens": 16
  }
}


()

## Putting the pieces together

You now have the two halves of the loop:

- **Request** = `messages` (system + user + history) **+** `tools` (what the
  model may call).
- **Response** = either a final answer (`stop`) or a set of **`tool_calls`**
  that the loop must execute and feed back.

That "feed back and repeat" is precisely what demo 2 makes concrete (executing
that single call, then many in parallel), demo 3 grows the toolset, and demos
`04`/`05` observe the whole loop with hooks.
